# BERT polarization classifier — v2 (first-25%-of-sentences, custom head)

Changes vs. the original `03_BERT.ipynb`, and why:

1. **Input truncation.** The dataset is already pre-cut upstream to the first ~25% of sentences
   per article (the most indicative part for polarization), so this notebook does **not**
   truncate again — it consumes `source_text` as-is. A lightweight QC cell just checks that the
   incoming text really does look short/pre-cut (flags it otherwise) before tokenizing.
2. **Text column choice.** Of `full_text` (clean text with entities masked), `bert_text`
   (mask-only) and `raw_version` (untouched raw text), the notebook defaults to `full_text`.
   Reasoning: the dataset is built from **topic triplets** (2 polarized + 1 neutral article per
   topic). If the model is trained on raw text, with ~137 documents it has every incentive to
   key off *which entities/parties are named* rather than *how the piece talks about them* —
   that's a shortcut, not polarization detection, and it won't generalize to new topics.
   `full_text`'s entity masking removes that shortcut while keeping full sentence structure
   (so BERT can still model syntax/framing). `bert_text` (mask-only) likely throws away too
   much surface text to let BERT learn stylistic cues; `raw_version` reintroduces the entity
   leakage. This is a judgment call in the absence of the actual CSV — the column choice is a
   single variable (`TEXT_COL_OVERRIDE`) below so you can flip it and rerun if your data looks
   different from this description.
3. **Sliding window.** Only wired in conditionally: after the 25%-sentence cut, we measure how
   many documents still exceed 512 tokens. If it's a small fraction, plain truncation is used
   (simplest, no aggregation logic needed). If it's non-trivial, documents are split into
   overlapping chunks, each chunk trains as its own example (label inherited from the parent
   article), and at eval time chunk-level probabilities are averaged back up to the
   article level before scoring. This only activates automatically if actually needed, so it
   doesn't add compute cost when it isn't.
4. **Classifier head.** Standard `BertForSequenceClassification` classifies off a single
   `[CLS]`-derived pooled vector. Here the head is a small MLP over
   `concat([CLS] vector, masked mean-pooled token vector)`, with `LayerNorm` + higher dropout
   (0.3). Mean pooling brings in signal from the whole (truncated) sequence rather than relying
   entirely on `[CLS]`, and the extra dropout is cheap insurance against overfitting on a
   dataset this small.
5. **Fold/topic integrity check.** Since triplets share a topic, a quick assertion confirms no
   `topic_id` is split across folds (otherwise train/eval leak through shared topic vocabulary).
6. **Grid search kept intentionally tiny**: 2 hyperparameter configs evaluated on a single
   validation fold (not full 5×4 nested CV), with early stopping, before the final fit on all
   data. This is meant to be re-run cheaply on modest hardware.

7. **Test-set source.** The held-out test set is no longer a reserved fold of the
   training CSV — it is loaded from a separate file (`TEST_CSV_PATH`). All folds in
   `CSV_PATH` are used for the k-fold grid search and the final fit; `TEST_CSV_PATH` is
   only touched once, in the final evaluation cell.


In [1]:
# Imports
import os
import re
import math
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import transformers
from transformers import (
    AutoTokenizer, BertModel, BertConfig, PreTrainedModel,
    DataCollatorWithPadding, TrainingArguments, Trainer, TrainerCallback,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.utils import logging as hf_logging
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

hf_logging.set_verbosity_error()

In [2]:
# Setup, reproducibility, device (unchanged from v1)
SEED = 123

def set_all_seeds(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[device] CUDA active -> {torch.cuda.get_device_name(0)}")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("[device] Apple MPS active")
    else:
        dev = torch.device("cpu")
        print("[device] No GPU found -> CPU (BERT will be slow)")
    return dev

device = get_device()
print(f"[versions] torch={torch.__version__} | transformers={transformers.__version__} | CUDA build={torch.version.cuda}")

[device] CUDA active -> Tesla T4
[versions] torch=2.11.0+cu128 | transformers=5.12.1 | CUDA build=12.8


## 1. Load data, pick the text column

Column-name detection is defensive: the CSV may name things `full_text`/`bert_text`/`text_bert`/`raw_version` etc. depending on which pipeline step produced it. Set `TEXT_COL_OVERRIDE` to force a specific choice; otherwise the priority list below is used.

In [4]:
CSV_PATH = r"..\Dati\Processed\dataset_processed_quantile1_sentences.csv"
#CSV_PATH = r"/content/dataset_processed_quantile1_sentences.csv"

# External test-set file. The held-out test set now comes from this separate file
# instead of being carved out of CSV_PATH via a reserved fold (see cell below, section 4).
# Same expected columns as CSV_PATH, except a 'fold' column is NOT required here — the
# whole file is treated as test data.
TEST_CSV_PATH = r"..\Dati\Processed\test_holdout_processed.csv"   # <-- set this to the external test-set file

MODEL_NAME = "bert-base-cased"   # cased: casing/punctuation are a stylistic (polarization) signal

# Candidate names for each conceptual column, in case of naming drift across pipeline steps
COLUMN_ALIASES = {
    "full_text":   ["full_text", "text_full", "clean_masked_text"],
    "bert_text":   ["bert_text", "text_bert"],
    "raw_version": ["raw_version", "raw_text", "text_raw"],
}
ID_COLS_TRAIN = ["article_id", "topic_id", "binary_label", "fold"]
ID_COLS_TEST  = ["article_id", "topic_id", "binary_label"]   # no 'fold' expected in the test file

# Priority order if TEXT_COL_OVERRIDE is not set — see markdown above for the reasoning
TEXT_COL_OVERRIDE = None   # e.g. force with "bert_text" or "raw_version" to A/B test
PRIORITY = ["raw_version", "bert_text", "bert_text"]

def resolve_column(canonical_name, df):
    for alias in COLUMN_ALIASES[canonical_name]:
        if alias in df.columns:
            return alias
    return None

def resolve_text_column(df_raw, label):
    available = {name: resolve_column(name, df_raw) for name in PRIORITY}
    print(f"[{label}] [columns found] {available}")

    if TEXT_COL_OVERRIDE is not None:
        chosen_canonical = TEXT_COL_OVERRIDE
    else:
        chosen_canonical = next((name for name in PRIORITY if available[name] is not None), None)

    if chosen_canonical is None or available[chosen_canonical] is None:
        raise ValueError(
            f"[{label}] Could not find a usable text column. Looked for {COLUMN_ALIASES}. "
            f"Actual columns in CSV: {list(df_raw.columns)}"
        )

    text_col = available[chosen_canonical]
    print(f"[{label}] [text column] using '{text_col}' (canonical: '{chosen_canonical}')")
    return text_col

def load_frame(csv_path, id_cols, label, has_fold):
    df_raw = pd.read_csv(csv_path)
    text_col = resolve_text_column(df_raw, label)

    missing_id_cols = [c for c in id_cols if c not in df_raw.columns]
    if missing_id_cols:
        raise ValueError(f"[{label}] Missing expected id/label columns: {missing_id_cols}")

    out = df_raw[id_cols + [text_col]].copy().reset_index(drop=True)
    out = out.rename(columns={text_col: "source_text"})
    if not has_fold:
        out["fold"] = -1   # placeholder: downstream code expects a 'fold' column, but it is
                            # never used to filter the external test set

    print(f"[{label}] [shape] {out.shape[0]} docs, {out.shape[1]} cols")
    print(f"[{label}] [NaN]\n{out.isna().sum()}")
    print(f"[{label}] [dup article_id] {out['article_id'].duplicated().sum()}")
    print(f"[{label}] [label balance overall]")
    print(out["binary_label"].value_counts(normalize=True).round(3).to_dict())
    return out, text_col

# ---- Train/val pool: used for grid search (k-fold CV over 'fold') and the final fit ----
df, TEXT_COL = load_frame(CSV_PATH, ID_COLS_TRAIN, label="train", has_fold=True)
print(f"\n[train] [folds] {sorted(df['fold'].unique())}")
print("\n[train] [label balance per fold]")
print(df.groupby("fold")["binary_label"].value_counts(normalize=True).round(2).unstack())
print("\n[train] [docs per fold]")
print(df["fold"].value_counts().sort_index().to_dict())
print("\n[train] [topics] unique topic_id count:", df["topic_id"].nunique(),
      "-> expect ~n_docs/3 if triplets are complete")

# ---- Held-out test set: now loaded from a separate file (TEST_CSV_PATH), NOT a reserved
# fold of CSV_PATH. All folds in df are used for training/validation. ----
df_test, TEXT_COL_TEST = load_frame(TEST_CSV_PATH, ID_COLS_TEST, label="test", has_fold=False)
if TEXT_COL_TEST != TEXT_COL:
    print(f"\n[WARNING] test set text column resolved to '{TEXT_COL_TEST}', "
          f"different from train's '{TEXT_COL}' — double-check the two files are consistent.")


[train] [columns found] {'raw_version': 'raw_text', 'bert_text': 'text_bert'}
[train] [text column] using 'raw_text' (canonical: 'raw_version')
[train] [shape] 978 docs, 5 cols
[train] [NaN]
article_id      0
topic_id        0
binary_label    0
fold            0
source_text     0
dtype: int64
[train] [dup article_id] 0
[train] [label balance overall]
{1: 0.667, 0: 0.333}

[train] [folds] [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

[train] [label balance per fold]
binary_label     0     1
fold                    
0             0.33  0.67
1             0.33  0.67
2             0.33  0.67
3             0.33  0.67
4             0.33  0.67

[train] [docs per fold]
{0: 195, 1: 195, 2: 198, 3: 195, 4: 195}

[train] [topics] unique topic_id count: 326 -> expect ~n_docs/3 if triplets are complete
[test] [columns found] {'raw_version': 'raw_text', 'bert_text': 'text_bert'}
[test] [text column] using 'raw_text' (canonical: 'raw_version')
[test] [shape] 69 docs, 5 cols
[test

## 2. Confirm the pre-truncation (no re-cutting here)

`source_text` already comes pre-cut to the first ~25% of sentences upstream, so this notebook does **not** truncate it again — `source_text_trunc` below is just an alias for clarity in later cells (naming kept so the rest of the pipeline reads the same either way). `split_sentences` is kept only as a lightweight, offline QC tool to sanity-check that the incoming text really is short (few sentences) and roughly consistent in length — a red flag here (e.g. articles with 40+ sentences) would suggest the upstream cut didn't apply to this column.

In [5]:
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-ZÀ-ÖØ-Þ0-9"\'])')

def split_sentences(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    text = re.sub(r"\s+", " ", text.strip())
    sents = _SENT_SPLIT_RE.split(text)
    return [s.strip() for s in sents if s.strip()]

LONG_DOC_THRESHOLD = 15  # heuristic: flag docs that look like they weren't actually pre-cut

def qc_pretruncation(frame, label):
    frame = frame.copy()
    frame["n_sentences"] = frame["source_text"].apply(lambda t: len(split_sentences(t)))
    frame["source_text_trunc"] = frame["source_text"]   # already pre-cut upstream — no further truncation

    print(f"[{label}] [QC] sentence counts in the incoming (already-cut) text")
    print(frame["n_sentences"].describe().round(1))
    suspicious = (frame["n_sentences"] > LONG_DOC_THRESHOLD).sum()
    if suspicious > 0:
        print(f"\n[{label}] [WARNING] {suspicious}/{len(frame)} docs have > {LONG_DOC_THRESHOLD} sentences — "
              f"double check that 'source_text' really is the pre-cut column.")
    else:
        print(f"\n[{label}] [ok] all docs have <= {LONG_DOC_THRESHOLD} sentences, consistent with a 25% pre-cut")
    return frame

df = qc_pretruncation(df, "train")
print()
df_test = qc_pretruncation(df_test, "test")


[train] [QC] sentence counts in the incoming (already-cut) text
count    978.0
mean       8.1
std        7.9
min        1.0
25%        4.0
50%        6.5
75%       10.0
max      109.0
Name: n_sentences, dtype: float64

[train] [WARNING] 65/978 docs have > 15 sentences — double check that 'source_text' really is the pre-cut column.

[test] [QC] sentence counts in the incoming (already-cut) text
count     69.0
mean      10.8
std       17.7
min        1.0
25%        4.0
50%        7.0
75%        9.0
max      116.0
Name: n_sentences, dtype: float64

[test] [WARNING] 6/69 docs have > 15 sentences — double check that 'source_text' really is the pre-cut column.


## 3. Token-length profile — decide if a sliding window is actually needed

In [6]:
MAX_LEN = 512

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok_lens = df["source_text_trunc"].apply(lambda t: len(tok(t, add_special_tokens=True)["input_ids"]))
df["_tok_len"] = tok_lens

print("[token length, pre-cut text] describe")
print(tok_lens.describe().round(1))
for thr in (128, 256, 384, 512):
    share = (tok_lens > thr).mean()
    print(f"  > {thr} tokens: {share:.1%} of docs")

OVERFLOW_SHARE = (tok_lens > MAX_LEN).mean()
NEED_SLIDING_WINDOW = OVERFLOW_SHARE > 0.05   # only bother if it affects a real chunk of the data
print(f"\n[decision] {OVERFLOW_SHARE:.1%} of docs still exceed {MAX_LEN} tokens -> "
      f"sliding window {'ENABLED' if NEED_SLIDING_WINDOW else 'not needed (plain truncation is enough)'}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[token length, pre-cut text] describe
count     978.0
mean      262.1
std       227.3
min        38.0
25%       132.0
50%       214.0
75%       326.8
max      2911.0
Name: source_text_trunc, dtype: float64
  > 128 tokens: 75.7% of docs
  > 256 tokens: 39.8% of docs
  > 384 tokens: 17.1% of docs
  > 512 tokens: 6.6% of docs

[decision] 6.6% of docs still exceed 512 tokens -> sliding window ENABLED


## 4. Optional sliding-window chunking

Only exercised if `NEED_SLIDING_WINDOW` is `True`. Each overflowing document is split into overlapping chunks (each chunk becomes its own training example, inheriting the parent label — a form of data augmentation for long docs); at eval time chunk-level probabilities for the same `article_id` are averaged back together before scoring, so metrics are still computed at the article level, not the chunk level. `max_chunks` is capped small to keep this cheap.

In [7]:
def chunk_encode(text, tokenizer, max_len=MAX_LEN, stride=64, max_chunks=2):
    """Overlapping-window tokenization for one document."""
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if not ids:
        return [tokenizer(text, truncation=True, max_length=max_len)]
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id
    window = max_len - 2  # leave room for [CLS]/[SEP]
    chunks, start = [], 0
    while start < len(ids) and len(chunks) < max_chunks:
        piece = ids[start:start + window]
        input_ids = [cls_id] + piece + [sep_id]
        enc = {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "token_type_ids": [0] * len(input_ids),
        }
        chunks.append(enc)
        if start + window >= len(ids):
            break
        start += window - stride
    return chunks

def build_examples(frame, tokenizer, max_len=MAX_LEN, use_sliding_window=False, stride=64, max_chunks=2):
    '''Row -> one or more model-ready examples. Always groupable back by article_id.'''
    records = []
    for row in frame.itertuples(index=False):
        text = row.source_text_trunc
        n_tokens = len(tokenizer(text, add_special_tokens=True)["input_ids"])
        if use_sliding_window and n_tokens > max_len:
            encs = chunk_encode(text, tokenizer, max_len=max_len, stride=stride, max_chunks=max_chunks)
        else:
            encs = [tokenizer(text, truncation=True, max_length=max_len)]
        for enc in encs:
            rec = {
                "input_ids": enc["input_ids"],
                "attention_mask": enc["attention_mask"],
                "labels": int(row.binary_label),
                "fold": int(row.fold),
                "article_id": row.article_id,
            }
            if "token_type_ids" in enc:
                rec["token_type_ids"] = enc["token_type_ids"]
            records.append(rec)
    return Dataset.from_list(records)


ds_trainval = build_examples(df, tok, max_len=MAX_LEN, use_sliding_window=NEED_SLIDING_WINDOW)
print(f"[dataset] {len(df)} articles -> {len(ds_trainval)} model examples "
      f"({'with' if NEED_SLIDING_WINDOW else 'without'} chunk expansion) [train+val pool]")

collator = DataCollatorWithPadding(tokenizer=tok)
MODEL_COLS = ["input_ids", "attention_mask", "token_type_ids", "labels"]

# ---- Held-out test set: loaded from a separate file (TEST_CSV_PATH) instead of being
# carved out of df via a reserved fold. Grid search (make_fold) and the final fit both
# operate on ds_trainval (all folds of CSV_PATH); ds_test is entirely separate data and is
# only touched in the final "held-out test evaluation" section at the bottom of the notebook. ----
ds_test = build_examples(df_test, tok, max_len=MAX_LEN, use_sliding_window=NEED_SLIDING_WINDOW)

print(f"[test split] external test file ({TEST_CSV_PATH}) used as held-out test: "
      f"{len(ds_test)} model examples / {len(set(ds_test['article_id']))} articles")
print(f"[test split] train+val pool (all folds of CSV_PATH): "
      f"{len(ds_trainval)} model examples / {len(set(ds_trainval['article_id']))} articles")


[dataset] 978 articles -> 1043 model examples (with chunk expansion) [train+val pool]
[test split] external test file (/content/test_holdout_processed.csv) used as held-out test: 75 model examples / 69 articles
[test split] train+val pool (all folds of CSV_PATH): 1043 model examples / 978 articles


## 5. Fold / topic-triplet integrity check

Since each `topic_id` produces a 2-polarized/1-neutral triplet, a topic split across folds would leak topic-specific vocabulary between train and eval. Quick assertion, not a full QC pass.

In [8]:
topic_fold_counts = df.groupby("topic_id")["fold"].nunique()
leaking_topics = topic_fold_counts[topic_fold_counts > 1]
if len(leaking_topics) > 0:
    print(f"[WARNING] {len(leaking_topics)} topic_id(s) span multiple folds — "
          f"train/eval leakage risk via shared topic vocabulary:")
    print(leaking_topics)
else:
    print("[ok] every topic_id sits entirely inside a single fold — no triplet leakage across folds")

triplet_sizes = df.groupby("topic_id").size()
print("\n[triplet size check] value counts (expect mostly 3):")
print(triplet_sizes.value_counts().sort_index())

[ok] every topic_id sits entirely inside a single fold — no triplet leakage across folds

[triplet size check] value counts (expect mostly 3):
3    326
Name: count, dtype: int64


## 6. Per-fold split builder + article-level prediction aggregation

`aggregate_predictions` groups any chunk-level probabilities back up to one row per `article_id` before scoring — this is a no-op when sliding-window chunking wasn't needed (one row already equals one article), so the same code path works either way.

In [9]:
def make_fold(k):
    folds = ds_trainval["fold"]  # grid search only ever sees the train+val pool
    train_idx = [i for i, f in enumerate(folds) if f != k]
    eval_idx  = [i for i, f in enumerate(folds) if f == k]

    train_ds = ds_trainval.select(train_idx)
    eval_ds  = ds_trainval.select(eval_idx)

    eval_ids    = list(eval_ds["article_id"])
    eval_labels_by_id = (
        pd.DataFrame({"article_id": eval_ids, "labels": eval_ds["labels"]})
        .drop_duplicates("article_id")
        .set_index("article_id")["labels"]
    )

    drop = [c for c in train_ds.column_names if c not in MODEL_COLS]
    train_ds = train_ds.remove_columns(drop)
    eval_ds  = eval_ds.remove_columns(drop)

    return train_ds, eval_ds, eval_ids, eval_labels_by_id


def aggregate_predictions(logits, article_ids):
    '''Mean-pool softmax probabilities across chunks of the same article_id,
    return (article_ids_unique, hard_preds, prob_class1).'''
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    tmp = pd.DataFrame({"article_id": article_ids, "p0": probs[:, 0], "p1": probs[:, 1]})
    agg = tmp.groupby("article_id", sort=False)[["p0", "p1"]].mean()
    preds = (agg["p1"] > agg["p0"]).astype(int)
    return agg.index.tolist(), preds.values, agg["p1"].values

In [10]:
# Sanity check on one fold
tr, ev, ids, y_by_id = make_fold(0)
print(f"[fold 0] train={len(tr)} model examples  eval={len(ev)} model examples "
      f"({len(y_by_id)} unique articles)")
print(f"[fold 0] model cols = {tr.column_names}")
print(f"[fold 0] eval class balance = {y_by_id.value_counts(normalize=True).round(2).to_dict()}")

[fold 0] train=837 model examples  eval=206 model examples (195 unique articles)
[fold 0] model cols = ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
[fold 0] eval class balance = {1: 0.67, 0: 0.33}


## 7. Model: BERT + custom pooling/classifier head, weighted loss, metrics

`BertPolarizationClassifier` pools `concat([CLS], masked-mean(tokens))` instead of the default `[CLS]`-only pooler, then runs it through a small MLP head (`Linear -> GELU -> LayerNorm -> Dropout(0.3) -> Linear`). Mean pooling uses signal from the whole truncated sequence; the extra dropout guards against overfitting on ~137 documents. `base_model_prefix = "bert"` lets `from_pretrained` correctly map the public `bert-base-cased` checkpoint weights onto `self.bert`.

In [11]:
FREEZE_LAYERS = 0        # 0 = full fine-tuning. Set e.g. 6 for a frozen-lower-layers ablation.
CLASSIFIER_DROPOUT = 0.36  # higher than the transformers default (0.1) — small-data regularization

class BertPolarizationClassifier(PreTrainedModel):
    config_class = BertConfig
    base_model_prefix = "bert"

    def __init__(self, config, dropout: float = CLASSIFIER_DROPOUT):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        hidden = config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
            nn.Linear(hidden, config.num_labels),
        )
        self.num_labels = config.num_labels
        self.post_init()

    def _pool(self, last_hidden_state, attention_mask):
        cls_vec = last_hidden_state[:, 0]
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(1)
        counts = mask.sum(1).clamp(min=1e-9)
        mean_vec = summed / counts
        return torch.cat([cls_vec, mean_vec], dim=-1)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = self._pool(outputs.last_hidden_state, attention_mask)
        logits = self.head(pooled)
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return SequenceClassifierOutput(
            loss=loss, logits=logits,
            hidden_states=outputs.hidden_states, attentions=outputs.attentions,
        )


def build_model(freeze_layers: int = FREEZE_LAYERS, dropout: float = CLASSIFIER_DROPOUT):
    torch.manual_seed(SEED)
    config = BertConfig.from_pretrained(MODEL_NAME, num_labels=2)
    model = BertPolarizationClassifier.from_pretrained(MODEL_NAME, config=config, dropout=dropout)

    if freeze_layers > 0:
        for p in model.bert.embeddings.parameters():
            p.requires_grad = False
        for layer in model.bert.encoder.layer[:freeze_layers]:
            for p in layer.parameters():
                p.requires_grad = False
    return model


def fold_class_weights(train_labels, damp=0.85):
    w = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array(train_labels))
    w = w ** damp
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float)


class WeightedTrainer(Trainer):
    '''Trainer with class-weighted CrossEntropy. Weights set per fold before training.'''
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


def compute_metrics_from_ids(logits, article_ids, y_by_id):
    '''Article-level metrics via aggregate_predictions (identity op when there's no chunking).'''
    ids_unique, preds, _ = aggregate_predictions(logits, article_ids)
    y_true = [y_by_id[i] for i in ids_unique]
    return {
        "f1_macro": f1_score(y_true, preds, average="macro"),
        "accuracy": accuracy_score(y_true, preds),
        "f1_class0": f1_score(y_true, preds, pos_label=0),
        "f1_class1": f1_score(y_true, preds, pos_label=1),
    }


# Quick check: weights on fold 0's train split
tr0, _, _, _ = make_fold(0)
w0 = fold_class_weights(tr0["labels"])
print(f"[fold 0] class weights (0,1) = {w0.tolist()}")

[fold 0] class weights (0,1) = [1.3094885349273682, 0.6905114054679871]


## 8. Light grid search — one validation fold, tiny grid, early stopping

Deliberately not a full 5-fold nested CV: 2 hyperparameter configs, evaluated on a single held-out fold, with early stopping so bad configs don't run to the full epoch budget. Pick the winner by macro-F1, then fit that config on all the data in the next section.

In [12]:
GRID = [(2e-5, 12)]   # (learning_rate, max_epochs) — small on purpose
VALIDATION_FOLD = [0,1]
TRAIN_BS, EVAL_BS = 8, 16

class EarlyStoppingOnF1(TrainerCallback):
    '''Stops if eval f1_macro doesn't improve for `patience` evals (cheap overfit guard).'''
    def __init__(self, patience=3):
        self.patience = patience
        self.best = -1.0
        self.bad_evals = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        score = metrics.get("eval_f1_macro", -1.0)
        if score > self.best:
            self.best = score
            self.bad_evals = 0
        else:
            self.bad_evals += 1
            if self.bad_evals >= self.patience:
                control.should_training_stop = True
        return control


def grid_args(lr, epochs):
    return TrainingArguments(
        output_dir="./_tmp_bert_grid",
        learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=0.01, warmup_ratio=0.1, seed=SEED,
        eval_strategy="epoch", save_strategy="no", logging_strategy="no",
        disable_tqdm=True, report_to="none",
        fp16=torch.cuda.is_available(),
    )

def grid_args(lr, epochs):
    return TrainingArguments(
        output_dir="./_tmp_bert_grid",
        learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=0.01, warmup_ratio=0.1, seed=SEED,
        eval_strategy="epoch", save_strategy="epoch",   # was "no" — must match eval_strategy
        save_total_limit=1,                              # keep disk usage light
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",                # Trainer prepends "eval_" itself
        greater_is_better=True,
        logging_strategy="no",
        disable_tqdm=True, report_to="none",
        fp16=torch.cuda.is_available(),
    )


def run_config(lr, epochs, val_folds=VALIDATION_FOLD, damp=0.5):
    all_fold_metrics = []
    for val_fold_idx in val_folds:
        print(f"  Running fold {val_fold_idx} for lr={lr}, epochs={epochs}")
        train_ds, eval_ds, eval_ids, y_by_id = make_fold(val_fold_idx)
        model = build_model()
        trainer = WeightedTrainer(
            model=model, args=grid_args(lr, epochs),
            train_dataset=train_ds, eval_dataset=eval_ds,
            data_collator=collator,
            compute_metrics=lambda ep: compute_metrics_from_ids(ep.predictions, eval_ids, y_by_id),
            class_weights=fold_class_weights(train_ds["labels"], damp=damp),
            callbacks=[EarlyStoppingOnF1(patience=2)],
        )
        trainer.train()
        logits = trainer.predict(eval_ds).predictions
        metrics = compute_metrics_from_ids(logits, eval_ids, y_by_id)
        all_fold_metrics.append(metrics)
        del model, trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not all_fold_metrics:
        return {"f1_macro": -1.0, "accuracy": -1.0, "f1_class0": -1.0, "f1_class1": -1.0}

    avg_metrics = {}
    for key in all_fold_metrics[0].keys():
        avg_metrics[key] = np.mean([m[key] for m in all_fold_metrics])
    return avg_metrics


results = []
for lr, epochs in GRID:
    m = run_config(lr, epochs)
    print(f"[grid] lr={lr} epochs={epochs} -> f1_macro={m['f1_macro']:.3f} acc={m['accuracy']:.3f}")
    results.append({"lr": lr, "epochs": epochs, **m})

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
print("\n[grid results]")
print(results_df)

BEST_LR, BEST_EPOCHS = results_df.iloc[0][["lr", "epochs"]]
print(f"\n[selected] lr={BEST_LR}  epochs={int(BEST_EPOCHS)}")

  Running fold 0 for lr=2e-05, epochs=12


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'eval_loss': '0.7792', 'eval_f1_macro': '0.3684', 'eval_accuracy': '0.4051', 'eval_f1_class0': '0.5207', 'eval_f1_class1': '0.2162', 'eval_runtime': '2.389', 'eval_samples_per_second': '86.24', 'eval_steps_per_second': '5.442', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'eval_loss': '2.035', 'eval_f1_macro': '0.7278', 'eval_accuracy': '0.7436', 'eval_f1_class0': '0.6622', 'eval_f1_class1': '0.7934', 'eval_runtime': '3.218', 'eval_samples_per_second': '63.09', 'eval_steps_per_second': '4.04', 'epoch': '8'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '2.042', 'eval_f1_macro': '0.7358', 'eval_accuracy': '0.7538', 'eval_f1_class0': '0.6667', 'eval_f1_class1': '0.8049', 'eval_runtime': '2.956', 'eval_samples_per_second': '68.67', 'eval_steps_per_second': '4.398', 'epoch': '9'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '810.9', 'train_samples_per_second': '12.43', 'train_steps_per_second': '1.554', 'train_loss': '0.2488', 'epoch': '9'}
[grid] lr=2e-05 epochs=12 -> f1_macro=0.768 acc=0.790

[grid results]
        lr  epochs  f1_macro  accuracy  f1_class0  f1_class1
0  0.00002      12  0.768471  0.789744   0.698383    0.83856

[selected] lr=2e-05  epochs=12


## 9. Final fit on the full internal dataset, save for external OOD test

In [13]:
FINAL_LR, FINAL_EPOCHS = float(BEST_LR), int(BEST_EPOCHS)
SAVE_DIR = "./bert_m02_v2_final"

# Fit on the train+val pool only (ds_trainval, i.e. all folds of CSV_PATH) — the held-out
# test set (ds_test, from TEST_CSV_PATH) is a separate file and stays completely unseen
# until the evaluation cell below.
full_ds = ds_trainval.remove_columns([c for c in ds_trainval.column_names if c not in MODEL_COLS])

final_args = TrainingArguments(
    output_dir="./_tmp_bert_final",
    learning_rate=FINAL_LR, num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    weight_decay=0.01, warmup_ratio=0.1, seed=SEED,
    eval_strategy="no", save_strategy="no", logging_strategy="no",
    disable_tqdm=True, report_to="none",
    fp16=torch.cuda.is_available(),
)

final = WeightedTrainer(
    model=build_model(),
    args=final_args,
    train_dataset=full_ds,
    data_collator=collator,
    class_weights=fold_class_weights(full_ds["labels"], damp=0.5),
)
final.train()

final.save_model(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)

import json
run_config_meta = {
    "text_column_used": TEXT_COL,
    "pretruncated_upstream": True,
    "max_len": MAX_LEN,
    "sliding_window_enabled": bool(NEED_SLIDING_WINDOW),
    "classifier_dropout": CLASSIFIER_DROPOUT,
    "final_lr": FINAL_LR,
    "final_epochs": FINAL_EPOCHS,
    "grid_results": results,
}
with open(os.path.join(SAVE_DIR, "run_config.json"), "w") as f:
    json.dump(run_config_meta, f, indent=2)

print(f"[saved] {SAVE_DIR}")
print(json.dumps(run_config_meta, indent=2))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'train_runtime': '484.1', 'train_samples_per_second': '25.85', 'train_steps_per_second': '3.247', 'train_loss': '0.1983', 'epoch': '12'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[saved] ./bert_m02_v2_final
{
  "text_column_used": "raw_text",
  "pretruncated_upstream": true,
  "max_len": 512,
  "sliding_window_enabled": true,
  "classifier_dropout": 0.36,
  "final_lr": 2e-05,
  "final_epochs": 12,
  "grid_results": [
    {
      "lr": 2e-05,
      "epochs": 12,
      "f1_macro": 0.7684714017638785,
      "accuracy": 0.7897435897435897,
      "f1_class0": 0.698382647265646,
      "f1_class1": 0.8385601562621111
    }
  ]
}


## 10. Held-out test evaluation

`ds_test` (fold `TEST_FOLD`, split off right after `ds_full` was built) has not been touched by the grid search or by the final fit — it was excluded from `ds_trainval` the whole way through. This is the only place it's used: run the final model on it once and report article-level metrics (via the same `aggregate_predictions` / `compute_metrics_from_ids` helpers used during CV, so chunked docs are handled the same way), plus a full classification report and confusion matrix.

In [14]:
test_ds = ds_test.remove_columns([c for c in ds_test.column_names if c not in MODEL_COLS])
test_article_ids = ds_test["article_id"]
test_labels_by_id = (
    pd.DataFrame({"article_id": test_article_ids, "labels": ds_test["labels"]})
    .drop_duplicates("article_id")
    .set_index("article_id")["labels"]
)

test_logits = final.predict(test_ds).predictions
test_metrics = compute_metrics_from_ids(test_logits, test_article_ids, test_labels_by_id)

print(f"[held-out test | external file: {TEST_CSV_PATH}] n_articles={len(test_labels_by_id)}")
print(json.dumps(test_metrics, indent=2))

test_ids_unique, test_preds, test_prob1 = aggregate_predictions(test_logits, test_article_ids)
test_y_true = [test_labels_by_id[i] for i in test_ids_unique]

from sklearn.metrics import classification_report, confusion_matrix
print("\n[classification report]")
print(classification_report(test_y_true, test_preds, target_names=["class0", "class1"], digits=3))
print("[confusion matrix] (rows=true, cols=pred)")
print(confusion_matrix(test_y_true, test_preds))

# Persist test results alongside the rest of the run config saved earlier
run_config_meta["test_source"] = TEST_CSV_PATH
run_config_meta["test_metrics"] = test_metrics
with open(os.path.join(SAVE_DIR, "run_config.json"), "w") as f:
    json.dump(run_config_meta, f, indent=2)
print(f"\n[saved] test metrics appended to {os.path.join(SAVE_DIR, 'run_config.json')}")

[held-out test | external file: /content/test_holdout_processed.csv] n_articles=69
{
  "f1_macro": 0.7943132309103417,
  "accuracy": 0.8115942028985508,
  "f1_class0": 0.7346938775510204,
  "f1_class1": 0.8539325842696629
}

[classification report]
              precision    recall  f1-score   support

      class0      0.692     0.783     0.735        23
      class1      0.884     0.826     0.854        46

    accuracy                          0.812        69
   macro avg      0.788     0.804     0.794        69
weighted avg      0.820     0.812     0.814        69

[confusion matrix] (rows=true, cols=pred)
[[18  5]
 [ 8 38]]

[saved] test metrics appended to ./bert_m02_v2_final/run_config.json
